# LangfuseEvaluationDataset — E2E Tests

Manual end-to-end scenarios for `LangfuseEvaluationDataset`. Run all cells
top-to-bottom and verify results in the **Langfuse UI** after each section.

**Prerequisites:**
- Langfuse credentials in `conf/local/credentials.yml`
- `langfuse>=3.14.0` installed

---

### Table of contents

| Section | Scenarios | What it covers |
|---------|-----------|----------------|
| **1. Local mode** | 1–4 | Fresh start, idempotent reload, incremental sync, `save()` with upsert |
| **2. Remote mode** | 5–7 | Remote load, `save()` without local file, versioned snapshots |
| **3. Edge cases & validation** | 8–11 | Items without `id`, metadata passing, config conflicts, invalid credentials |
| **4. Lifecycle & integration** | 12–13 | Native API update/archive/delete, running an experiment |

---
# 1. Local Mode (Scenarios 1–4)

## Setup

In [ ]:
import json
import logging
import sys
import tempfile
from datetime import datetime
from pathlib import Path

import yaml

# Ensure the src directory is importable when running from notebooks/
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = str(PROJECT_ROOT / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from kedro_agentic_workflows.datasets.langfuse_evaluation_dataset import (
    LangfuseEvaluationDataset,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(name)s — %(levelname)s — %(message)s",
)

# Load credentials from the Kedro local config
CREDENTIALS_FILE = Path("../conf/local/credentials.yml")
with open(CREDENTIALS_FILE) as f:
    all_credentials = yaml.safe_load(f)

CREDENTIALS = {
    "public_key": all_credentials["langfuse_credentials"]["public_key"],
    "secret_key": all_credentials["langfuse_credentials"]["secret_key"],
    "host": all_credentials["langfuse_credentials"]["host"],
}

# Unique name per run — avoids cleanup issues
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
DATASET_NAME = f"e2e-local-{RUN_ID}"

TMP_DIR = Path(tempfile.mkdtemp(prefix="e2e_langfuse_"))
LOCAL_FILE = TMP_DIR / "eval_items.json"

print(f"Dataset name : {DATASET_NAME}")
print(f"Local file   : {LOCAL_FILE}")

## Test data

Synthetic evaluation items reused across all four scenarios.

In [ ]:
INITIAL_ITEMS = [
    {
        "id": "e2e_001",
        "input": {"question": "How do I reset my password?"},
        "expected_output": {"intent": "account_management"},
    },
    {
        "id": "e2e_002",
        "input": {"question": "My order hasn't arrived yet."},
        "expected_output": {"intent": "order_status"},
    },
    {
        "id": "e2e_003",
        "input": {"question": "Can I get a refund?"},
        "expected_output": {"intent": "refund_request"},
    },
]

# Single new item added in Scenario 3
NEW_ITEM = {
    "id": "e2e_004",
    "input": {"question": "I want to cancel my subscription."},
    "expected_output": {"intent": "cancellation"},
}

# Items passed to save() in Scenario 4:
#   e2e_005 — new
#   e2e_006 — new
#   e2e_001 — existing id with UPDATED content (will be upserted)
SAVE_ITEMS = [
    {
        "id": "e2e_005",
        "input": {"question": "Where is your nearest store?"},
        "expected_output": {"intent": "store_locator"},
    },
    {
        "id": "e2e_006",
        "input": {"question": "Do you ship internationally?"},
        "expected_output": {"intent": "shipping_info"},
    },
    {
        "id": "e2e_001",
        "input": {"question": "How do I change my password?"},
        "expected_output": {"intent": "account_management_v2"},
    },
]

print("Test data defined ✓")

## Helpers

In [ ]:
def write_local(items: list[dict]) -> None:
    """Write items to the local JSON test file."""
    LOCAL_FILE.write_text(json.dumps(items, indent=2))
    print(f"  Wrote {len(items)} item(s) to local file")


def read_local() -> list[dict]:
    """Read items from the local JSON test file."""
    return json.loads(LOCAL_FILE.read_text())


def make_dataset() -> LangfuseEvaluationDataset:
    """Create a dataset instance with the shared test config."""
    return LangfuseEvaluationDataset(
        dataset_name=DATASET_NAME,
        credentials=CREDENTIALS,
        filepath=str(LOCAL_FILE),
        sync_policy="local",
    )


def summarise_remote(dataset_client) -> dict:
    """Return a summary dict of the remote dataset for inspection."""
    return {
        "count": len(dataset_client.items),
        "ids": sorted(item.id for item in dataset_client.items),
    }

---
## Scenario 1: Fresh start — create remote dataset + sync local items

**Steps:**
1. Write 3 items (`e2e_001`, `e2e_002`, `e2e_003`) to the local JSON file.
2. Instantiate `LangfuseEvaluationDataset` with `sync_policy="local"`.
3. Call `load()`.

**Expected behaviour:**
- Remote dataset is created (it didn't exist before).
- All 3 local items are upserted to remote (all new on first run).
- `load()` returns a `DatasetClient` with 3 items.

**Expected logs:**
```
Dataset 'e2e-local-...' not found on Langfuse, creating it.
Upserting 3 item(s) from '...' to remote dataset 'e2e-local-...'.
Loaded dataset 'e2e-local-...' with 3 item(s) (sync_policy='local').
```

**Verify in Langfuse UI:**
- Dataset `e2e-local-<timestamp>` exists under Datasets.
- Contains exactly 3 items: `e2e_001`, `e2e_002`, `e2e_003`.
- Each item has the correct `input` and `expected_output`.

In [ ]:
print("=" * 60)
print("SCENARIO 1: Fresh start")
print("=" * 60)

# Step 1: write local file
write_local(INITIAL_ITEMS)

# Step 2-3: create dataset + load
ds = make_dataset()
result = ds.load()

# Assertions
summary = summarise_remote(result)
assert summary["count"] == 3, f"Expected 3 items, got {summary['count']}"
assert summary["ids"] == ["e2e_001", "e2e_002", "e2e_003"], (
    f"Unexpected ids: {summary['ids']}"
)

print(f"\n✓ PASSED — Remote has {summary['count']} items: {summary['ids']}")
print("  → Check Langfuse UI: dataset should exist with 3 items")

---
## Scenario 2: Idempotent reload — no duplicates on repeated `load()`

**Steps:**
1. Create a new `LangfuseEvaluationDataset` instance (same config as Scenario 1).
2. Call `load()` again without changing the local file.

**Expected behaviour:**
- All 3 local items are upserted again. Since content is unchanged,
  `create_dataset_item()` updates each item to the same values (idempotent no-op).
- `load()` returns a `DatasetClient` with exactly 3 items — no duplicates.

**Expected logs:**
```
Upserting 3 item(s) from '...' to remote dataset 'e2e-local-...'.
Loaded dataset 'e2e-local-...' with 3 item(s) (sync_policy='local').
```

**Verify in Langfuse UI:**
- Still exactly 3 items (not 6).

In [ ]:
print("=" * 60)
print("SCENARIO 2: Idempotent reload")
print("=" * 60)

# Same local file, fresh dataset instance
ds2 = make_dataset()
result2 = ds2.load()

# Assertions
summary2 = summarise_remote(result2)
assert summary2["count"] == 3, f"Expected 3 items, got {summary2['count']}"
assert summary2["ids"] == ["e2e_001", "e2e_002", "e2e_003"], (
    f"Unexpected ids: {summary2['ids']}"
)

print(f"\n✓ PASSED — Still {summary2['count']} items (no duplicates)")
print("  → Check Langfuse UI: still exactly 3 items, no duplicates")

---
## Scenario 3: Incremental sync — one new local item

**Steps:**
1. Append a new item (`e2e_004`) to the local file (so it now has 4 items).
2. Create a new `LangfuseEvaluationDataset` instance and call `load()`.

**Expected behaviour:**
- All 4 local items are upserted. The 3 existing items are updated to the same
  values (no-op). The new item (`e2e_004`) is created.
- `load()` returns a `DatasetClient` with 4 items.

**Expected logs:**
```
Upserting 4 item(s) from '...' to remote dataset 'e2e-local-...'.
Loaded dataset 'e2e-local-...' with 4 item(s) (sync_policy='local').
```

**Verify in Langfuse UI:**
- 4 items total: `e2e_001` – `e2e_004`.
- `e2e_001` – `e2e_003` have their original content (upsert with same data is a no-op).

In [ ]:
print("=" * 60)
print("SCENARIO 3: Incremental sync")
print("=" * 60)

# Step 1: add one new item to local file
write_local(INITIAL_ITEMS + [NEW_ITEM])

# Step 2: load
ds3 = make_dataset()
result3 = ds3.load()

# Assertions
summary3 = summarise_remote(result3)
assert summary3["count"] == 4, f"Expected 4 items, got {summary3['count']}"
assert "e2e_004" in summary3["ids"], "New item e2e_004 not found on remote"

# Verify original item content is unchanged
item_001 = next(i for i in result3.items if i.id == "e2e_001")
assert item_001.input["question"] == "How do I reset my password?", (
    f"Original item was modified: {item_001.input}"
)

print(f"\n✓ PASSED — {summary3['count']} items: {summary3['ids']}")
print(f"  e2e_001 content preserved: '{item_001.input['question']}'")
print("  → Check Langfuse UI: 4 items, original content intact")

---
## Scenario 4: `save()` — upsert all items (create + update), merge to local

**Steps:**
1. Call `save(SAVE_ITEMS)` where `SAVE_ITEMS` contains:
   - `e2e_005` — new item (created)
   - `e2e_006` — new item (created)
   - `e2e_001` — existing id with **updated content** (upserted → updated on remote)
2. Call `load()` to get the final remote state.
3. Read the local file to verify the merge.

**Expected behaviour — remote:**
- All 3 items are upserted: `e2e_005` and `e2e_006` are created, `e2e_001` is
  **updated** with the new content ("How do I change my password?").
- Total: 6 items on remote.

**Expected behaviour — local file:**
- Merged content: 6 items total (4 existing + 2 new).
- `e2e_001` is **replaced** by the new version from `SAVE_ITEMS`
  (new items take precedence in `_merge_items`).

**Expected logs:**
```
Upserting 3 item(s) to remote dataset 'e2e-local-...'.
```

**Verify in Langfuse UI:**
- 6 items: `e2e_001` – `e2e_006`.
- `e2e_001` now shows "How do I change my password?" (updated via upsert).

In [ ]:
print("=" * 60)
print("SCENARIO 4: save() with upsert (create + update)")
print("=" * 60)

# Step 1: save 2 new + 1 update
ds4 = make_dataset()
ds4.save(SAVE_ITEMS)

# Step 2: reload remote to verify
result4 = ds4.load()
summary4 = summarise_remote(result4)

expected_ids = ["e2e_001", "e2e_002", "e2e_003", "e2e_004", "e2e_005", "e2e_006"]
assert summary4["count"] == 6, f"Expected 6 remote items, got {summary4['count']}"
assert summary4["ids"] == expected_ids, f"Unexpected remote ids: {summary4['ids']}"

# Verify e2e_001 was UPDATED on remote (upsert replaced content)
item_001_remote = next(i for i in result4.items if i.id == "e2e_001")
assert item_001_remote.input["question"] == "How do I change my password?", (
    f"Remote e2e_001 should be updated! Got: {item_001_remote.input}"
)
print(f"  Remote: {summary4['count']} items, e2e_001 updated ✓")

# Step 3: verify local file
local_items = read_local()
local_ids = sorted(item["id"] for item in local_items)
assert len(local_items) == 6, f"Expected 6 local items, got {len(local_items)}"
assert local_ids == expected_ids, f"Unexpected local ids: {local_ids}"

# Verify e2e_001 was UPDATED in local file (new takes precedence)
item_001_local = next(i for i in local_items if i["id"] == "e2e_001")
assert item_001_local["input"]["question"] == "How do I change my password?", (
    f"Local e2e_001 should be updated! Got: {item_001_local['input']}"
)
print(f"  Local:  {len(local_items)} items, e2e_001 updated ✓")

print(f"\n✓ PASSED — Remote and local both have 6 items, e2e_001 updated via upsert")
print("  → Check Langfuse UI: 6 items, e2e_001 shows 'How do I change my password?'")

### Section 1 checklist

| Check | Expected |
|-------|----------|
| Dataset `e2e-local-<ts>` exists | Yes |
| Total items | 6 (`e2e_001` – `e2e_006`) |
| `e2e_001` input | `{"question": "How do I change my password?"}` (updated in Scenario 4) |
| No duplicates | Each id appears exactly once |

---
# 2. Remote Mode (Scenarios 5–7)

These scenarios test `sync_policy="remote"`. They **reuse the dataset created
by Scenarios 1–4** (6 items on Langfuse at this point).

| Scenario | What it tests |
|----------|---------------|
| 5 | Load from existing remote — no local file interaction |
| 6 | `save()` in remote mode — uploads to remote, no local file written |
| 7 | Versioned load — pin to a historical snapshot |

## Scenario 5: Remote mode — load from existing dataset

**Steps:**
1. Instantiate `LangfuseEvaluationDataset` with `sync_policy="remote"` and **no `filepath`**.
2. Call `load()`.

**Expected behaviour:**
- No dataset creation (it already exists from Scenarios 1–4).
- No local file interaction at all.
- `load()` returns a `DatasetClient` with all 6 items from the remote dataset.

**Expected logs:**
```
Loaded dataset 'e2e-local-...' with 6 item(s) (sync_policy='remote').
```
No "Syncing" or "creating" log lines.

**Verify in Langfuse UI:**
- Dataset unchanged — still 6 items.

In [ ]:
print("=" * 60)
print("SCENARIO 5: Remote mode — load from existing dataset")
print("=" * 60)

ds5 = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME,
    credentials=CREDENTIALS,
    sync_policy="remote",
)

result5 = ds5.load()
summary5 = summarise_remote(result5)

assert summary5["count"] == 6, f"Expected 6 items, got {summary5['count']}"
expected_ids = ["e2e_001", "e2e_002", "e2e_003", "e2e_004", "e2e_005", "e2e_006"]
assert summary5["ids"] == expected_ids, f"Unexpected ids: {summary5['ids']}"

# Spot-check content
item_003 = next(i for i in result5.items if i.id == "e2e_003")
assert item_003.input["question"] == "Can I get a refund?", (
    f"Unexpected content for e2e_003: {item_003.input}"
)

print(f"\n✓ PASSED — Remote mode loaded {summary5['count']} items: {summary5['ids']}")
print("  No local file was read or written")
print("  → Check Langfuse UI: dataset unchanged, still 6 items")

---
## Scenario 6: Remote mode — `save()` uploads to remote, no local file

**Steps:**
1. Instantiate `LangfuseEvaluationDataset` with `sync_policy="remote"` (no `filepath`).
2. Call `save()` with a new item.
3. Call `load()` to verify the item was uploaded to remote.
4. Verify the local file was NOT modified (still has 6 items from Scenario 4).

**Expected behaviour:**
- `save()` upserts the new item to remote.
- No local file is read or written.
- `load()` returns 7 items (6 existing + 1 new).

**Expected logs:**
```
Upserting 1 item(s) to remote dataset 'e2e-local-...'.
```

**Verify in Langfuse UI:**
- 7 items total — includes the new `e2e_remote_save_001`.

In [ ]:
print("=" * 60)
print("SCENARIO 6: Remote mode — save() uploads to remote, no local file")
print("=" * 60)

REMOTE_SAVE_ITEMS = [
    {
        "id": "e2e_remote_save_001",
        "input": {"question": "Can I change my delivery address?"},
        "expected_output": {"intent": "delivery_update"},
    },
]

ds6 = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME,
    credentials=CREDENTIALS,
    sync_policy="remote",
)

# save() should upload to remote
ds6.save(REMOTE_SAVE_ITEMS)

# Verify the item was uploaded to remote
result6 = ds6.load()
summary6 = summarise_remote(result6)

assert summary6["count"] == 7, f"Expected 7 items, got {summary6['count']}"
assert "e2e_remote_save_001" in summary6["ids"], (
    "e2e_remote_save_001 was NOT uploaded to remote"
)

# Verify local file was NOT modified (still has 6 items from Scenario 4)
local_items_after = read_local()
local_ids_after = sorted(item["id"] for item in local_items_after)
assert len(local_items_after) == 6, (
    f"Local file should still have 6 items, got {len(local_items_after)}"
)
assert "e2e_remote_save_001" not in local_ids_after, (
    "e2e_remote_save_001 should NOT be in local file"
)

print(f"\n✓ PASSED — Remote has {summary6['count']} items (new item uploaded)")
print(f"  Local file unchanged: {len(local_items_after)} items")
print("  → Check Langfuse UI: 7 items, e2e_remote_save_001 present")

---
## Scenario 7: Remote mode — versioned load

**Precondition:** 7 items on remote after Scenario 6.

**Steps:**
1. Record a timestamp **before** adding a new item.
2. Add a new item (`e2e_version_001`) directly via the Langfuse client to the
   existing remote dataset.
3. Record a timestamp **after** the addition.
4. Load with `version` set to the **before** timestamp — should NOT include the new item.
5. Load with `version` set to the **after** timestamp — should include the new item.
6. Load without `version` — should return latest (includes the new item).

**Expected behaviour:**
- Versioned load at the "before" timestamp returns 7 items (the new item didn't exist yet).
- Versioned load at the "after" timestamp returns 8 items.
- Unversioned load returns 8 items (latest state).

**Note:** This scenario requires `langfuse>=3.14.0`. If the SDK version is older,
the versioned `get_dataset` call will fail.

**Verify in Langfuse UI:**
- 8 items total after this scenario (the new `e2e_version_001` is visible).

In [ ]:
import time
from datetime import timezone

print("=" * 60)
print("SCENARIO 7: Remote mode — versioned load")
print("=" * 60)

# Step 1: record timestamp BEFORE adding the new item
ts_before = datetime.now(tz=timezone.utc)
print(f"  Timestamp before: {ts_before.isoformat()}")

# Small delay to ensure timestamp separation
time.sleep(2)

# Step 2: add a new item directly via the Langfuse client
from langfuse import Langfuse

lf_client = Langfuse(**CREDENTIALS)
lf_client.create_dataset_item(
    dataset_name=DATASET_NAME,
    id="e2e_version_001",
    input={"question": "What are your opening hours?"},
    expected_output={"intent": "business_hours"},
)
lf_client.flush()
print("  Added e2e_version_001 to remote dataset")

# Small delay to let Langfuse process the item
time.sleep(2)

# Step 3: record timestamp AFTER the addition
ts_after = datetime.now(tz=timezone.utc)
print(f"  Timestamp after:  {ts_after.isoformat()}")

# Step 4: versioned load at "before" timestamp — should have 6 items
ds7_before = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME,
    credentials=CREDENTIALS,
    sync_policy="remote",
    version=ts_before.isoformat(),
)
result7_before = ds7_before.load()
summary7_before = summarise_remote(result7_before)

print(f"\n  Versioned load (before): {summary7_before['count']} items")
assert summary7_before["count"] == 7, (
    f"Expected 7 items at 'before' timestamp, got {summary7_before['count']}"
)
assert "e2e_version_001" not in summary7_before["ids"], (
    "e2e_version_001 should NOT be in the 'before' snapshot"
)
print("  ✓ 'before' snapshot has 7 items (new item excluded)")

# Step 5: versioned load at "after" timestamp — should have 8 items
ds7_after = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME,
    credentials=CREDENTIALS,
    sync_policy="remote",
    version=ts_after.isoformat(),
)
result7_after = ds7_after.load()
summary7_after = summarise_remote(result7_after)

print(f"  Versioned load (after):  {summary7_after['count']} items")
assert summary7_after["count"] == 8, (
    f"Expected 8 items at 'after' timestamp, got {summary7_after['count']}"
)
assert "e2e_version_001" in summary7_after["ids"], (
    "e2e_version_001 should be in the 'after' snapshot"
)
print("  ✓ 'after' snapshot has 8 items (new item included)")

# Step 6: unversioned load — should return latest (8 items)
ds7_latest = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME,
    credentials=CREDENTIALS,
    sync_policy="remote",
)
result7_latest = ds7_latest.load()
summary7_latest = summarise_remote(result7_latest)

print(f"  Unversioned load:        {summary7_latest['count']} items")
assert summary7_latest["count"] == 8, (
    f"Expected 8 items (latest), got {summary7_latest['count']}"
)

print(f"\n✓ PASSED — Versioned load works correctly")
print(f"  before={ts_before.isoformat()} → 7 items")
print(f"  after ={ts_after.isoformat()} → 8 items")
print(f"  latest → 8 items")
print("  → Check Langfuse UI: 8 items total, e2e_version_001 present")

### Section 2 checklist

| Check | Expected |
|-------|----------|
| Total items | 8 (6 from local + `e2e_remote_save_001` + `e2e_version_001`) |
| `e2e_remote_save_001` present | Uploaded by `save()` in remote mode (Scenario 6) |
| `e2e_version_001` present | Added in Scenario 7 |
| Local file unchanged | Still 6 items (no remote-mode writes) |
| Versioned snapshots | Before-timestamp excludes `e2e_version_001` |

---
# 3. Edge Cases & Validation (Scenarios 8–11)

| Scenario | What it tests |
|----------|---------------|
| 8 | Items without `id` — always uploaded, warning logged, dedup works for items with `id` |
| 9 | Metadata passing — user `metadata` reaches Langfuse on dataset creation |
| 10 | Config conflict — `version` + `sync_policy="local"` raises `DatasetError` |
| 11 | Invalid credentials — missing or empty credentials raise `DatasetError` |

## Scenario 8: Items without `id` — dedup edge case

**Steps:**
1. Create a fresh dataset (new unique name to isolate this test).
2. Call `save()` with a mix of items: 2 with `id`, 1 without `id`.
3. Call `save()` again with the same data.
4. Call `load()` and inspect remote state.

**Expected behaviour:**
- First `save()`: all 3 items upserted (2 with id created + 1 without id created).
- Second `save()`: all 3 items upserted again. Items with `id` are updated
  to the same values (idempotent no-op). The item without `id` creates a new
  entry (cannot be deduplicated). A warning is logged about the id-less item.
- Remote ends up with 4 items: 2 with id (no duplicates) + 2 without id
  (each `save()` creates a new entry for items without id).

**Expected logs (on each `save()`):**
```
Found 1 item(s) without an 'id' field. Items without 'id' cannot be deduplicated ...
```

In [ ]:
print("=" * 60)
print("SCENARIO 8: Items without id — dedup edge case")
print("=" * 60)

DATASET_NAME_S8 = f"e2e-no-id-{RUN_ID}"

MIXED_ITEMS = [
    {
        "id": "e2e_with_id_001",
        "input": {"question": "Item with id — first"},
        "expected_output": {"intent": "has_id"},
    },
    {
        "id": "e2e_with_id_002",
        "input": {"question": "Item with id — second"},
        "expected_output": {"intent": "has_id"},
    },
    {
        "input": {"question": "Item WITHOUT id — should be re-uploaded every time"},
        "expected_output": {"intent": "no_id"},
    },
]

ds8 = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME_S8,
    credentials=CREDENTIALS,
    sync_policy="remote",
)

# First save: all 3 items uploaded
ds8.save(MIXED_ITEMS)
result8a = ds8.load()
summary8a = summarise_remote(result8a)
print(f"  After first save: {summary8a['count']} items")
assert summary8a["count"] == 3, f"Expected 3 items after first save, got {summary8a['count']}"

# Second save: items with id upserted (no-op), id-less item creates a new entry
ds8.save(MIXED_ITEMS)
result8b = ds8.load()
summary8b = summarise_remote(result8b)
print(f"  After second save: {summary8b['count']} items")
assert summary8b["count"] == 4, f"Expected 4 items after second save, got {summary8b['count']}"

# Verify: 2 items with known ids + 2 duplicates of the id-less item
ids_with_id = [item.id for item in result8b.items if item.id in ("e2e_with_id_001", "e2e_with_id_002")]
assert len(ids_with_id) == 2, f"Expected 2 items with known ids, got {len(ids_with_id)}"

items_no_id_content = [
    item for item in result8b.items
    if item.input.get("question") == "Item WITHOUT id — should be re-uploaded every time"
]
assert len(items_no_id_content) == 2, (
    f"Expected 2 copies of the id-less item, got {len(items_no_id_content)}"
)

print(f"\n✓ PASSED — Items with id deduplicated, item without id re-uploaded")
print(f"  {len(ids_with_id)} items with id (no duplicates)")
print(f"  {len(items_no_id_content)} copies of id-less item (re-uploaded as expected)")
print(f"  → Check Langfuse UI: dataset '{DATASET_NAME_S8}' has 4 items")

---
## Scenario 9: Metadata passing on dataset creation

**Steps:**
1. Create a new dataset with custom `metadata`.
2. Call `load()` to trigger dataset creation on Langfuse.
3. Fetch the dataset directly via the Langfuse client and inspect its metadata.

**Expected behaviour:**
- The remote dataset's metadata matches the dict passed at init time.

**Verify in Langfuse UI:**
- Open the dataset and check the metadata section shows the custom fields.

In [ ]:
print("=" * 60)
print("SCENARIO 9: Metadata passing on dataset creation")
print("=" * 60)

DATASET_NAME_S9 = f"e2e-metadata-{RUN_ID}"
CUSTOM_METADATA = {
    "project": "intent-detection",
    "team": "ml-platform",
    "environment": "test",
}

ds9 = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME_S9,
    credentials=CREDENTIALS,
    sync_policy="remote",
    metadata=CUSTOM_METADATA,
)

# load() triggers dataset creation
ds9.load()

# Fetch the dataset directly to inspect metadata
from langfuse import Langfuse

lf_client = Langfuse(**CREDENTIALS)
remote_dataset = lf_client.api.datasets.get(dataset_name=DATASET_NAME_S9)

print(f"  Remote metadata: {remote_dataset.metadata}")
for key, value in CUSTOM_METADATA.items():
    assert key in remote_dataset.metadata, f"Missing metadata key: '{key}'"
    assert remote_dataset.metadata[key] == value, (
        f"Metadata '{key}' mismatch: expected '{value}', got '{remote_dataset.metadata[key]}'"
    )

print(f"\n✓ PASSED — All custom metadata fields present on remote dataset")
print(f"  → Check Langfuse UI: dataset '{DATASET_NAME_S9}' metadata shows custom fields")

---
## Scenario 10: Config conflict — `version` with `sync_policy="local"`

**Steps:**
1. Attempt to create a `LangfuseEvaluationDataset` with `version` set and
   `sync_policy="local"`.

**Expected behaviour:**
- `DatasetError` is raised at init time with a message explaining the conflict.

In [ ]:
from kedro.io import DatasetError

print("=" * 60)
print("SCENARIO 10: Config conflict — version + local sync policy")
print("=" * 60)

try:
    LangfuseEvaluationDataset(
        dataset_name="should-not-be-created",
        credentials=CREDENTIALS,
        sync_policy="local",
        version="2026-01-15T00:00:00Z",
    )
    assert False, "Should have raised DatasetError"
except DatasetError as e:
    print(f"  Caught expected error: {e}")
    assert "version" in str(e).lower(), f"Error message should mention 'version': {e}"
    assert "remote" in str(e).lower(), f"Error message should mention 'remote': {e}"

print(f"\n✓ PASSED — DatasetError raised for version + local conflict")

---
## Scenario 11: Invalid credentials

**Steps:**
1. Attempt to create with missing `public_key`.
2. Attempt to create with empty `secret_key`.
3. Attempt to create with invalid `sync_policy`.
4. Attempt to create with unsupported file extension.

**Expected behaviour:**
- Each case raises `DatasetError` at init time with a descriptive message.

In [ ]:
print("=" * 60)
print("SCENARIO 11: Invalid credentials and config")
print("=" * 60)

test_cases = [
    {
        "label": "Missing public_key",
        "kwargs": {
            "dataset_name": "x",
            "credentials": {"secret_key": "sk-test"},
        },
        "expect_in_error": "public_key",
    },
    {
        "label": "Empty secret_key",
        "kwargs": {
            "dataset_name": "x",
            "credentials": {"public_key": "pk-test", "secret_key": "  "},
        },
        "expect_in_error": "secret_key",
    },
    {
        "label": "Invalid sync_policy",
        "kwargs": {
            "dataset_name": "x",
            "credentials": {"public_key": "pk-test", "secret_key": "sk-test"},
            "sync_policy": "invalid",
        },
        "expect_in_error": "sync_policy",
    },
    {
        "label": "Unsupported file extension",
        "kwargs": {
            "dataset_name": "x",
            "credentials": {"public_key": "pk-test", "secret_key": "sk-test"},
            "filepath": "data/eval.csv",
        },
        "expect_in_error": ".csv",
    },
]

for tc in test_cases:
    try:
        LangfuseEvaluationDataset(**tc["kwargs"])
        assert False, f"  ✗ {tc['label']}: should have raised DatasetError"
    except DatasetError as e:
        assert tc["expect_in_error"] in str(e), (
            f"  ✗ {tc['label']}: error should mention '{tc['expect_in_error']}': {e}"
        )
        print(f"  ✓ {tc['label']}: {e}")

print(f"\n✓ PASSED — All {len(test_cases)} validation cases raised DatasetError")

### Section 3 checklist

| Check | Expected |
|-------|----------|
| Dataset `e2e-no-id-<ts>` | 4 items (2 with id + 2 copies of id-less item) |
| Dataset `e2e-metadata-<ts>` | Metadata shows `project`, `team`, `environment` |
| No dataset `should-not-be-created` | Config conflict caught at init |

---
# 4. Lifecycle & Integration (Scenarios 12–13)

These scenarios validate the **delegation strategy** (lifecycle operations
managed via native Langfuse API) and the end-to-end experiment workflow.

| Scenario | What it tests |
|----------|---------------|
| 12 | Lifecycle via native API — update, archive, delete items, then reload |
| 13 | Running an experiment — `DatasetClient.run_experiment()` works end-to-end |

## Scenario 12: Lifecycle via native Langfuse API

This scenario validates that lifecycle operations performed through the native
Langfuse API are transparently reflected when our dataset reloads. This confirms
the **delegation strategy**: `LangfuseEvaluationDataset` handles load/save, while
update, archive, and delete are delegated to the native API.

**Steps:**
1. Create a fresh dataset with 3 items via `save()`.
2. **Update** one item's `expected_output` via native `create_dataset_item()` (upsert by id).
3. **Archive** another item by setting `status="ARCHIVED"` via native API.
4. **Delete** the third item via `api.dataset_items.delete()`.
5. Reload via `LangfuseEvaluationDataset.load()` and verify all changes are reflected.

**Expected behaviour:**
- Updated item has new `expected_output`.
- Archived item has `status == "ARCHIVED"`.
- Deleted item is gone entirely.

**Verify in Langfuse UI:**
- Updated item shows new content.
- Archived item is marked as archived.
- Deleted item is no longer listed.

In [ ]:
print("=" * 60)
print("SCENARIO 12: Lifecycle via native Langfuse API")
print("=" * 60)

from langfuse import Langfuse

DATASET_NAME_S12 = f"e2e-lifecycle-{RUN_ID}"

LIFECYCLE_ITEMS = [
    {
        "id": "lc_to_update",
        "input": {"question": "What is your return policy?"},
        "expected_output": {"intent": "returns"},
    },
    {
        "id": "lc_to_archive",
        "input": {"question": "How do I track my order?"},
        "expected_output": {"intent": "order_tracking"},
    },
    {
        "id": "lc_to_delete",
        "input": {"question": "This item will be deleted."},
        "expected_output": {"intent": "to_be_removed"},
    },
]

# Step 1: create dataset with 3 items
ds12 = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME_S12,
    credentials=CREDENTIALS,
    sync_policy="remote",
)
ds12.save(LIFECYCLE_ITEMS)

result12_before = ds12.load()
assert len(result12_before.items) == 3, f"Expected 3 items, got {len(result12_before.items)}"
print(f"  Created dataset with {len(result12_before.items)} items")

# Step 2: UPDATE — change expected_output via native upsert
lf = Langfuse(**CREDENTIALS)
lf.create_dataset_item(
    dataset_name=DATASET_NAME_S12,
    id="lc_to_update",
    input={"question": "What is your return policy?"},
    expected_output={"intent": "return_policy_updated"},
)
lf.flush()
print("  Updated lc_to_update via native API")

# Step 3: ARCHIVE — set status to ARCHIVED
lf.create_dataset_item(
    dataset_name=DATASET_NAME_S12,
    id="lc_to_archive",
    input={"question": "How do I track my order?"},
    expected_output={"intent": "order_tracking"},
    status="ARCHIVED",
)
lf.flush()
print("  Archived lc_to_archive via native API")

# Step 4: DELETE — permanently remove item
item_to_delete = next(i for i in result12_before.items if i.id == "lc_to_delete")
lf.api.dataset_items.delete(id=item_to_delete.id)
print("  Deleted lc_to_delete via native API")

# Step 5: reload via our dataset and verify
import time
time.sleep(2)

result12_after = ds12.load()
summary12 = summarise_remote(result12_after)
print(f"\n  After lifecycle operations: {summary12['count']} items, ids: {summary12['ids']}")

# Verify UPDATE
item_updated = next((i for i in result12_after.items if i.id == "lc_to_update"), None)
assert item_updated is not None, "lc_to_update should still exist"
assert item_updated.expected_output["intent"] == "return_policy_updated", (
    f"Expected updated output, got: {item_updated.expected_output}"
)
print("  ✓ lc_to_update: expected_output updated correctly")

# Verify ARCHIVE
item_archived = next((i for i in result12_after.items if i.id == "lc_to_archive"), None)
assert item_archived is not None, "lc_to_archive should still exist"
assert item_archived.status == "ARCHIVED", (
    f"Expected status ARCHIVED, got: {item_archived.status}"
)
print("  ✓ lc_to_archive: status is ARCHIVED")

# Verify DELETE
assert "lc_to_delete" not in summary12["ids"], (
    "lc_to_delete should have been permanently deleted"
)
print("  ✓ lc_to_delete: permanently deleted")

print(f"\n✓ PASSED — All lifecycle operations reflected after reload")
print(f"  → Check Langfuse UI: dataset '{DATASET_NAME_S12}'")

---
## Scenario 13: Running an experiment

This scenario proves the `DatasetClient` returned by `load()` can be used to
run a real experiment via `run_experiment()`.

**Steps:**
1. Load an existing dataset (reuses the lifecycle dataset from Scenario 12).
2. Define a simple task function that echoes the input.
3. Call `dataset.run_experiment()` with that task.
4. Verify the experiment run completes and is visible.

**Expected behaviour:**
- `run_experiment()` returns without error.
- The experiment run is visible in the Langfuse UI under the dataset.

**Verify in Langfuse UI:**
- A new experiment run appears under the dataset with results for each item.

In [ ]:
print("=" * 60)
print("SCENARIO 13: Running an experiment")
print("=" * 60)

from langfuse import Langfuse

EXPERIMENT_NAME = f"e2e-experiment-{RUN_ID}"

# Step 1: load a dataset with active items
ds13 = LangfuseEvaluationDataset(
    dataset_name=DATASET_NAME_S12,
    credentials=CREDENTIALS,
    sync_policy="remote",
)
dataset_client = ds13.load()

active_items = [i for i in dataset_client.items if i.status != "ARCHIVED"]
print(f"  Dataset has {len(active_items)} active item(s)")

# Step 2: define a simple task function (no @observe — it can cause hangs with run_experiment)
def echo_task(item):
    """Simple task that echoes the input as the 'prediction'."""
    return {"prediction": item.input.get("question", "unknown")}

# Step 3: run the experiment
experiment_run = dataset_client.run_experiment(
    name=EXPERIMENT_NAME,
    task=echo_task,
)

lf = Langfuse(**CREDENTIALS)
lf.flush()

print(f"  Experiment '{EXPERIMENT_NAME}' completed")

print(f"\n✓ PASSED — Experiment run completed successfully")
print(f"  → Check Langfuse UI: experiment '{EXPERIMENT_NAME}' under dataset '{DATASET_NAME_S12}'")

### Section 4 checklist

| Check | Expected |
|-------|----------|
| Dataset `e2e-lifecycle-<ts>` | 2 items (1 deleted) |
| `lc_to_update` expected_output | `{"intent": "return_policy_updated"}` |
| `lc_to_archive` status | `ARCHIVED` |
| `lc_to_delete` | Gone (permanently deleted) |
| Experiment `e2e-experiment-<ts>` | Visible under dataset with run results |